# Limits of first- and second-order schemes

In [ ]:
#    APM41012EP course notebook - Chapter 11 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    PDE: Part II: Limits of first- and second-order schemes
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "seaborn"

In [ ]:
def upwind(u, nt, a, dtoverdx):
    for it in range(nt-1):
        flux = a*u
        u[1:] += -dtoverdx * (flux[1:] - flux[:-1])
        u[0] = u[-1]
    return u

def lax_wendroff(u, nt, a, dtoverdx):
    nu = a*dtoverdx
    for it in range(nt-1):        
        flux = u[:-1] + ((1-nu)/2)*(u[1:] - u[:-1])
        u[1:-1] += -dtoverdx * (flux[1:] - flux[:-1])
        u[0] = u[-2]
        u[-1] = u[0]
    return u

## Linear advection equation

We consider the linear advection equation:

$$ \partial_t u(x,t) + a \, \partial_x u(x,t) = 0$$

We take $a=1$ as the propagation speed.

### First-order scheme for smooth initial data

We choose smooth initial data:

$$ u(x,0) = u_0(x) = sin(2 \pi x) $$

In [ ]:
xmin =  0.0
xmax =  1.0

tini = 0.0
tend = 3.1

a = 1.

nx = 400
dx = (xmax-xmin) / nx
x = np.linspace(xmin+(dx/2), xmax-(dx/2), nx)

cfl = 0.1

nt = int(a*(tend-tini)/(cfl*dx)) + 1
dt = (tend-tini)/(nt-1)
dtoverdx = dt/dx

uini = lambda x : np.sin(2*np.pi*x)

# initialization of the solution with a ghost cell on the left and periodic boundary conditions
unum = np.zeros(nx+1)
unum[1:] = uini(x)
unum[0] = unum[-1]

unum = upwind(unum, nt, a, dtoverdx)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=uini(x), name='initial sol.'))
fig.add_trace(go.Scatter(x=x, y=uini(a*x-tend), name=f'exact sol. (t={tend})'))
fig.add_trace(go.Scatter(x=x, y=unum[1:], name=f'num. sol. (t={tend})'))
legend = dict(x=0.8, bgcolor='rgba(0,0,0,0)')
fig.update_layout(height=500, legend=legend)
fig.show() 

### Second-order scheme for smooth initial data

In [ ]:
xmin =  0.0
xmax =  1.0

tini = 0.0
tend = 1.1

a = 1.0

nx = 400
dx = (xmax-xmin) / nx
x = np.linspace(xmin+(dx/2), xmax-(dx/2), nx)

cfl = 0.5

nt = int(a*(tend-tini)/(cfl*dx)) + 1
dt = (tend-tini)/(nt-1)
dtoverdx = dt/dx

uini = lambda x : np.sin(2*np.pi*x)

# initialization of the solution with a ghost cell on the left and on the right and periodic boundary conditions
unum = np.zeros(nx+2)
unum[1:-1] = uini(x)
unum[0] = unum[-2]
unum[-1] = unum[0]

unum = lax_wendroff(unum, nt, a, dtoverdx)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=uini(x), name='initial sol.'))
fig.add_trace(go.Scatter(x=x, y=uini(a*x-tend), name=f'exact sol. (t={tend})'))
fig.add_trace(go.Scatter(x=x, y=unum[1:-1], name=f'num. sol. (t={tend})'))
legend = dict(x=0.8, bgcolor='rgba(0,0,0,0)')
fig.update_layout(height=500, legend=legend)
fig.show() 

### First-order scheme for discontinuous initial data

We choose discontinuous initial data of square-wave type.

In [ ]:
# Initial solution: square wave
def step_ini(x):
    u = np.empty(x.size)
    for ix, xi in enumerate(x):
        if xi <=-0.5 or xi>=0.5:
            u[ix] = 0.
        else:
            u[ix] = 1.
    return u

In [ ]:
xmin =  -2.0
xmax =  2.0

tini = 0.0
tend = 1.1

a = 1.0

nx = 400
dx = (xmax-xmin) / nx
x = np.linspace(xmin+(dx/2), xmax-(dx/2), nx)

cfl = 0.5

nt = int(a*(tend-tini)/(cfl*dx)) + 1
dt = (tend-tini)/(nt-1)
dtoverdx = dt/dx

# initialization of the solution with a ghost cell on the left and periodic boundary conditions
unum = np.zeros(nx+1)
unum[1:] = step_ini(x)
unum[0] = unum[-1]

unum = upwind(unum, nt, a, dtoverdx)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=step_ini(x), name='initial sol.'))
fig.add_trace(go.Scatter(x=x, y=step_ini(x-a*tend), name='exact sol.'))
fig.add_trace(go.Scatter(x=x, y=unum[1:], name='num. sol.'))
legend = dict(x=0.05, bgcolor='rgba(0,0,0,0)')
fig.update_layout(height=500, legend=legend)
fig.show()

### Second-order scheme for discontinuous initial data

In [ ]:
xmin =  -2.0
xmax =  2.0

tini = 0.0
tend = 1.1

a = 1.0

nx = 400
dx = (xmax-xmin) / nx
x = np.linspace(xmin+(dx/2), xmax-(dx/2), nx)

cfl = 0.5

nt = int(a*(tend-tini)/(cfl*dx)) + 1
dt = (tend-tini)/(nt-1)
dtoverdx = dt/dx

# initialization of the solution with a ghost cell on the left and periodic boundary conditions
unum = np.zeros(nx+2)
unum[1:-1] = step_ini(x)
unum[0] = unum[-2]
unum[-1] = unum[0]

unum = lax_wendroff(unum, nt, a, dtoverdx)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=step_ini(x), name='initial sol.'))
fig.add_trace(go.Scatter(x=x, y=step_ini(x-a*tend), name='exact sol.'))
fig.add_trace(go.Scatter(x=x, y=unum[1:-1], name='num. sol.'))
legend = dict(x=0.05, bgcolor='rgba(0,0,0,0)')
fig.update_layout(height=500, legend=legend)
fig.show()

The first-order scheme is very "diffusive" and requires a very large number of points to converge towards smooth or singular solutions. In the singular case, convergence of first-order methods is even slower than in the smooth case. Moving to second order with a scheme such as Lax-Wendroff is highly relevant for smooth solutions but raises serious problems when the solution is discontinuous. One must then analyze the notion of singular solution: we speak of solutions in the weak sense or in the sense of distributions, and build schemes with good stability properties.